In [207]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings

In [208]:
video_id = "eIrMbAQSU34"  # only the ID, not full URL

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id=video_id, languages=["en"])

    # Support both dict-style and object-style transcript snippets
    transcript = " ".join(
        chunk["text"] if isinstance(chunk, dict) else chunk.text
        for chunk in transcript_list
    )
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"Failed to fetch transcript: {e}")

Hi! My name is Mosh and I'm gonna be your
instructor in this Java course. In this course, you're gonna learn everything you
need to get started programming in Java. We'll start off by installing all the
necessary tools to build Java applications then you're gonna learn
about the basics of Java you'll learn how Java code gets executed you'll learn
how to build simple algorithms and throughout this course I'm gonna share
with you lots of tips and shortcuts from my years of experience I'll teach you
how to write good code like a professional developer so we'll end up
watching this course you will have a solid foundation in Java and be ready to
learn about advanced Java features I've designed this course for anyone who
wants to learn Java if you're a beginner don't worry I'll make Java super simple
and hold your hands through this entire course you're not too old or too young
you'll write your first Java program in minutes
my name is mosh I'm a software engineer with two decades of experie

In [209]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="Hi! My name is Mosh and I'm gonna be your\ninstructor in this Java course. In this", start=0.03, duration=3.96), FetchedTranscriptSnippet(text="course, you're gonna learn everything you\nneed to get started programming in Java.", start=3.99, duration=3.42), FetchedTranscriptSnippet(text="We'll start off by installing all the\nnecessary tools to build Java", start=7.41, duration=3.66), FetchedTranscriptSnippet(text="applications then you're gonna learn\nabout the basics of Java you'll learn", start=11.07, duration=3.63), FetchedTranscriptSnippet(text="how Java code gets executed you'll learn\nhow to build simple algorithms and", start=14.7, duration=4.71), FetchedTranscriptSnippet(text="throughout this course I'm gonna share\nwith you lots of tips and shortcuts from", start=19.41, duration=3.69), FetchedTranscriptSnippet(text="my years of experience I'll teach you\nhow to write good code like a", start=23.1, duration=3.54), Fetc

In [210]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [211]:
len(chunks)

161

In [212]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9063.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [213]:
vector_store.index_to_docstore_id

{0: '52f2444d-82a9-4125-90d0-0a8c0f1975bb',
 1: '9ca10178-98d4-41c6-83d7-4490b83e91b1',
 2: 'e680b2c9-605c-4776-b918-dadc721b12b0',
 3: '174b8a67-e669-4aa5-8bf7-b05efa0acd95',
 4: 'f8c3139c-cfe6-4edd-8bf3-3eb5552c883b',
 5: 'ab40732c-8a99-4d0e-813c-0f4054c6a9d4',
 6: 'f854e6f6-65e8-40d5-bbfe-0a6153bb4c5d',
 7: 'd21e799e-97f5-4f86-b0ca-1119ff21076d',
 8: 'ad0190af-a3f4-4973-a452-6f34c58c9d7c',
 9: 'e4bf0053-0900-4442-86a0-9650f49f3dcb',
 10: '51e4e322-33d5-4b68-bac9-52f46978e612',
 11: '0d40fb57-36c8-42c7-8263-ecd1ae0a7097',
 12: 'adbce877-d565-45f6-b0b5-41d0714aacc4',
 13: 'e25035b5-6f58-4d55-a3dc-16822eefba8d',
 14: 'ae6e9166-9fc4-4d8c-805b-b487c6af2a1e',
 15: '084a37a2-5b22-4473-a565-a097a2041c81',
 16: '1ec104fb-3173-4c91-ba96-56fb8229af32',
 17: '51cca2b5-564c-472c-ab35-81e1bd4ad3c8',
 18: 'acb74155-8fbe-4213-927f-94cb4c31f42e',
 19: 'f4dc2580-8709-474a-b578-a108aa1c4000',
 20: 'a9e8d5ba-d392-4df9-b369-d1dc8837a5d2',
 21: '8e68aedc-c38f-4b8f-aea4-8490404c07f5',
 22: '98fd3e60-db32-

In [214]:
vector_store.get_by_ids(['2296428f-8c0d-429a-8229-a475d1b8ca22'])

[]

In [215]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [216]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001BD12BA8160>, search_kwargs={'k': 4})

In [217]:
retriever.invoke('What is deepmind')

[Document(id='9ca10178-98d4-41c6-83d7-4490b83e91b1', metadata={}, page_content="and hold your hands through this entire course you're not too old or too young\nyou'll write your first Java program in minutes\nmy name is mosh I'm a software engineer with two decades of experience and I've\ntaught over 3 million people how to code and how to become professional software\nengineers I have a coding school at code with mass comm where you can find plenty\nof courses that help you take her coding skills to the next level I hope you'll\nstick around and learn this beautiful and powerful programming language and\nnow award from this video sponsor as someone who runs an online business I\ncannot stress enough the importance of staying safe online which is why I was\nso excited when dashlane reached out to me if you don't know - Lane is the\npassword manager and VPN recommended by Apple and Google and it's a fantastic\nsafeguard for keeping your information secure it's completely free to use for

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="just replace this with your groq api key"
)

In [219]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [220]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [221]:
retrieved_docs

[Document(id='f4dc2580-8709-474a-b578-a108aa1c4000', metadata={}, page_content="support me by liking and sharing this video on the social networks that you\nuse often also be sure to subscribe and enable the notifications so next time I\nupload a video you'll get notified thank you so much and let's continue watching all right now let's see what exactly\nhappens under the hood the moment we run a Java program in IntelliJ\nthere are basically two steps involved here compilation and execution in the\ncompilation step IntelliJ uses the Java compiler to compile our code into a\ndifferent format called Java bytecode this Java compiler comes with the Java\ndevelopment kit that we downloaded at the beginning of the course let me show\nyou so here we can right click on this main the Java and in this context menu\nwe have an item called open in terminal it's down below unfortunately it's not\nvisible in my recording window it's called open in terminal on Mac and\nprobably open in command prompt

In [222]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"support me by liking and sharing this video on the social networks that you\nuse often also be sure to subscribe and enable the notifications so next time I\nupload a video you'll get notified thank you so much and let's continue watching all right now let's see what exactly\nhappens under the hood the moment we run a Java program in IntelliJ\nthere are basically two steps involved here compilation and execution in the\ncompilation step IntelliJ uses the Java compiler to compile our code into a\ndifferent format called Java bytecode this Java compiler comes with the Java\ndevelopment kit that we downloaded at the beginning of the course let me show\nyou so here we can right click on this main the Java and in this context menu\nwe have an item called open in terminal it's down below unfortunately it's not\nvisible in my recording window it's called open in terminal on Mac and\nprobably open in command prompt on Windows so let's open that we get this\n\nwe're done with our first program

In [223]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [224]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      support me by liking and sharing this video on the social networks that you\nuse often also be sure to subscribe and enable the notifications so next time I\nupload a video you'll get notified thank you so much and let's continue watching all right now let's see what exactly\nhappens under the hood the moment we run a Java program in IntelliJ\nthere are basically two steps involved here compilation and execution in the\ncompilation step IntelliJ uses the Java compiler to compile our code into a\ndifferent format called Java bytecode this Java compiler comes with the Java\ndevelopment kit that we downloaded at the beginning of the course let me show\nyou so here we can right click on this main the Java and in this context menu\nwe have an item called open in terminal it's down below unfortunately it

In [225]:
answer = llm.invoke(final_prompt)
print(answer.content)

No, the topic of nuclear fusion is not discussed in this video. The video appears to be about Java programming, specifically how to run a Java program in IntelliJ and the compilation and execution process.


In [226]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [227]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [228]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [229]:
parallel_chain.invoke('who is Demis')

{'context': "variable called role and here we set this to admin now to check the role of\nthe user we can write an if statement like this if role equals admin then\nperhaps we want to print you are an admin now\nyou might be wondering why we have this condition here it's obvious that this\ncondition is always true because we have set roll to admin but this is just for\ndemonstration in a real program we are not gonna hard code this admin here so\nwe're gonna read the role of the current user from somewhere else\nwe don't know what it is at the time of writing code okay so here we have one\ncondition let's write another condition else if\nrole equals moderator perhaps we want to display a different message so you are a\nmoderator and finally if the role is none of these values you want to print\nyou are a guest so this is one way to implement this scenario using an if\nstatement we can also implement this using a switch statement and sometimes\n\nwe're dealing with two different types o

In [230]:
parser = StrOutputParser()

In [231]:
main_chain = parallel_chain | prompt | llm | parser

In [232]:
main_chain.invoke('which topic covered in this video')

'The topics covered in this video include:\n\n1. How Java code gets executed under the hood, specifically compilation and execution.\n2. Introduction to Java programming, including running a Java program in IntelliJ.\n3. Clean coding and its importance in programming.\n4. Control flow statements, including conditional statements and loops.\n5. The process of building and executing a Java application in IntelliJ.'